# Beauty Recommendation Chatbot
## Complete AI-Powered Beauty Assistant with Product Recommendations

This notebook implements a comprehensive beauty recommendation system that:
- Analyzes user concerns and preferences
- Provides personalized product recommendations
- Offers natural home remedies
- Includes a fully functional shopping cart


## Phase 1: Setup and Data Loading

In [ ]:
# Install required packages
!pip install ipywidgets pandas numpy matplotlib pillow requests nltk
!pip install --upgrade ipywidgets

# Enable widgets
from google.colab import output
output.enable_custom_widget_manager()

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import json
import re
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import matplotlib.pyplot as plt
from PIL import Image
import requests
from io import BytesIO
import nltk
from collections import defaultdict
import random

# Download NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

print("✅ All libraries imported successfully!")

### Data Loading and Validation

In [ ]:
# File upload widget for CSV files
from google.colab import files

print("📁 Please upload your CSV files:")
print("Expected files: products.csv, concerns.csv, ingredients.csv (or similar)")
print("\nClick 'Choose Files' below to upload:")

uploaded = files.upload()

# Display uploaded files
print(f"\n✅ Uploaded {len(uploaded)} file(s):")
for filename in uploaded.keys():
    print(f"  - {filename}")

In [ ]:
# Data loading and validation class
class DataLoader:
    def __init__(self):
        self.products_df = None
        self.concerns_df = None
        self.ingredients_df = None
        self.data_loaded = False
    
    def load_csv_files(self, uploaded_files):
        """Load and validate CSV files"""
        try:
            for filename, content in uploaded_files.items():
                df = pd.read_csv(BytesIO(content))
                
                # Auto-detect file type based on columns
                if 'product_id' in df.columns.str.lower():
                    self.products_df = df
                    print(f"✅ Loaded products data: {filename} ({len(df)} rows)")
                elif 'concern' in df.columns.str.lower():
                    self.concerns_df = df
                    print(f"✅ Loaded concerns data: {filename} ({len(df)} rows)")
                elif 'ingredient' in df.columns.str.lower():
                    self.ingredients_df = df
                    print(f"✅ Loaded ingredients data: {filename} ({len(df)} rows)")
                else:
                    # Try to load as general product data
                    self.products_df = df
                    print(f"✅ Loaded as product data: {filename} ({len(df)} rows)")
            
            self.validate_data()
            self.data_loaded = True
            return True
            
        except Exception as e:
            print(f"❌ Error loading CSV files: {str(e)}")
            return False
    
    def validate_data(self):
        """Validate loaded data structure"""
        print("\n🔍 Data Validation:")
        
        if self.products_df is not None:
            print(f"  Products DataFrame: {self.products_df.shape}")
            print(f"  Columns: {list(self.products_df.columns)}")
            
        if self.concerns_df is not None:
            print(f"  Concerns DataFrame: {self.concerns_df.shape}")
            print(f"  Columns: {list(self.concerns_df.columns)}")
            
        if self.ingredients_df is not None:
            print(f"  Ingredients DataFrame: {self.ingredients_df.shape}")
            print(f"  Columns: {list(self.ingredients_df.columns)}")
    
    def get_sample_data(self):
        """Display sample data for verification"""
        if self.products_df is not None:
            print("\n📋 Sample Product Data:")
            display(self.products_df.head())
        
        if self.concerns_df is not None:
            print("\n📋 Sample Concerns Data:")
            display(self.concerns_df.head())

# Initialize data loader
data_loader = DataLoader()
success = data_loader.load_csv_files(uploaded)

if success:
    print("\n🎉 Data loading completed successfully!")
    data_loader.get_sample_data()
else:
    print("\n❌ Data loading failed. Please check your CSV files.")

## Phase 2-7: Core Chatbot Implementation

In [ ]:
# Load the core chatbot logic
exec(open('beauty_chatbot_core.py').read())
print("✅ Core chatbot logic loaded!")

In [ ]:
# Load the UI components
exec(open('beauty_chatbot_ui.py').read())
print("✅ UI components loaded!")

### Initialize and Launch the Beauty Chatbot

In [ ]:
# Initialize the chatbot with loaded data
if data_loader.data_loaded:
    beauty_bot = BeautyChatbot(data_loader)
    chatbot_ui = BeautyChatbotUI(beauty_bot)
    
    print("🎉 Beauty Recommendation Chatbot is ready!")
    print("\n" + "="*50)
    
    # Display the complete interface
    chatbot_ui.display()
    
else:
    print("❌ Please load your CSV data first before initializing the chatbot.")

### Testing and Demo Section

In [ ]:
# Demo scenarios for testing
demo_messages = [
    "I have acne problems",
    "My skin is very dry and flaky",
    "Can you recommend some good skincare products?",
    "I'm looking for anti-aging solutions",
    "What natural remedies work for oily skin?",
    "Hello, how are you today?"
]

print("🧪 Demo Test Messages:")
for i, msg in enumerate(demo_messages, 1):
    print(f"{i}. {msg}")

print("\nYou can copy and paste these into the chat to test different conversation types!")

### Analytics and Insights

In [ ]:
# Analytics dashboard
def show_analytics():
    if not hasattr(beauty_bot, 'conversation_history'):
        print("No conversation data available yet.")
        return
    
    print("📊 Chatbot Analytics:")
    print(f"Total conversations: {len(beauty_bot.conversation_history)}")
    print(f"Items in cart: {len(beauty_bot.cart)}")
    
    if beauty_bot.cart:
        total_value = beauty_bot.get_cart_total()
        print(f"Cart total value: ${total_value:.2f}")
    
    if beauty_bot.user_profile:
        print(f"User profile: {beauty_bot.user_profile}")

# Create analytics button
analytics_btn = widgets.Button(
    description='Show Analytics 📊',
    button_style='info'
)

def show_analytics_handler(b):
    show_analytics()

analytics_btn.on_click(show_analytics_handler)
display(analytics_btn)

### Export and Save Functions

In [ ]:
# Export conversation history
def export_conversation():
    if hasattr(beauty_bot, 'conversation_history'):
        import json
        from datetime import datetime
        
        # Convert datetime objects to strings for JSON serialization
        exportable_history = []
        for entry in beauty_bot.conversation_history:
            export_entry = {}
            for key, value in entry.items():
                if isinstance(value, datetime):
                    export_entry[key] = value.isoformat()
                else:
                    export_entry[key] = str(value)
            exportable_history.append(export_entry)
        
        filename = f"beauty_chat_history_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
        
        with open(filename, 'w') as f:
            json.dump({
                'conversation_history': exportable_history,
                'user_profile': beauty_bot.user_profile,
                'cart': beauty_bot.cart
            }, f, indent=2)
        
        print(f"✅ Conversation exported to {filename}")
        
        # Download file in Colab
        from google.colab import files
        files.download(filename)
    else:
        print("No conversation history to export.")

export_btn = widgets.Button(
    description='Export Chat 💾',
    button_style='success'
)

export_btn.on_click(lambda b: export_conversation())
display(export_btn)

## Usage Instructions

### How to Use the Beauty Chatbot:

1. **Upload Your Data**: First, upload your CSV files containing product information, concerns mapping, and ingredients data.

2. **Set Your Profile**: Fill out the user profile section with your age, budget, skin type, etc. This helps personalize recommendations.

3. **Start Chatting**: Use the chat interface to:
   - Ask about specific concerns ("I have acne")
   - Request product recommendations ("Show me skincare products")
   - Get natural remedy suggestions
   - Have general beauty conversations

4. **Browse Products**: View recommended products with images, ratings, and reviews.

5. **Add to Cart**: Click "Add to Cart" on products you like.

6. **View Details**: Click "View Details" for comprehensive product information.

7. **Checkout**: Use the cart section to review items and checkout.

### Features:
- ✅ Concern-based recommendations
- ✅ Exploration-based product discovery
- ✅ Natural home remedies
- ✅ Fully functional shopping cart
- ✅ Product detail views
- ✅ User profile personalization
- ✅ Conversation export
- ✅ Analytics dashboard

### Tips:
- Be specific about your concerns for better recommendations
- Save your profile for personalized suggestions
- Try the demo messages to see different conversation types
- Export your chat history to save recommendations
